# Clase 08 — Usando el pipeline con datos nuevos
## Scoring de quiebra: de un modelo entrenado a un Excel para el negocio

---

El jefe nos manda un archivo con empresas nuevas y nos pide: **"¿cuáles están en riesgo?"**

- **No hay target:** todavía no sabemos quién quiebra. Para eso es el modelo.
- **No se entrena nada:** el pipeline ya está entrenado (`practica08.ipynb`, sección 7). Acá solo **cargamos y predecimos**.
- **Entregable:** un Excel con tres hojas, una por umbral (0.70, 0.35 y 0.50), con tres columnas: empresa, probabilidad de quiebra y clasificación.

In [1]:
import pandas as pd
import joblib

---
## 1. Cargar el pipeline y los datos nuevos

`joblib.load` devuelve el pipeline tal como quedó al entrenar: imputador y XGBoost, con todo lo aprendido adentro. No hay `.fit()` en todo este notebook.

In [2]:
pipeline = joblib.load("pipeline_xgboost.joblib")
datos = pd.read_csv("datos_nuevos.csv")

print(f"Empresas nuevas: {datos.shape[0]}")
datos[["id_empresa"]].head()

Empresas nuevas: 1000


,id_empresa
0,EMP-0001
1,EMP-0002
2,EMP-0003
3,EMP-0004
4,EMP-0005


**Observación:** el archivo trae una columna identificadora (`id_empresa`) y los 95 ratios, pero **no** `Bankrupt?`. El identificador no es una variable del modelo: lo usamos solo para saber de qué empresa habla cada fila.

In [3]:
columnas_modelo = list(pipeline.feature_names_in_)
X_nuevo = datos[columnas_modelo]   # mismas columnas y mismo orden que en el entrenamiento

print("Nulos en datos nuevos:", X_nuevo.isna().sum().sum())

Nulos en datos nuevos: 0


---
## 2. Predecir

`predict_proba` devuelve dos columnas: probabilidad de sana y de quiebra. Nos quedamos con la segunda.

In [4]:
datos["Probabilidad de positivo"] = pipeline.predict_proba(X_nuevo)[:, 1]

datos[["id_empresa", "Probabilidad de positivo"]].describe().round(3)

,Probabilidad de positivo
count,1000.000
mean,0.160
std,0.278
min,0.001
25%,0.006
50%,0.020
75%,0.150
max,0.993


**Observación:** la mayoría de las empresas tiene probabilidad baja y unas pocas tienen probabilidad alta. Es lo esperable con una clase positiva chica. El umbral decide cuántas de esas empresas terminan en la lista de riesgo.

---
## 3. Clasificar con tres umbrales

Misma probabilidad, tres políticas de riesgo: **0.70** (conservador, solo las más seguras), **0.50** (neutro) y **0.35** (agresivo, más empresas a revisar).

In [5]:
umbrales = [0.70, 0.35, 0.50]   # orden de las hojas del Excel
hojas = {}

for umbral in umbrales:
    hoja = pd.DataFrame({
        "Dato": datos["id_empresa"],
        "Probabilidad de positivo": datos["Probabilidad de positivo"].round(4),
        "Clasificación": (datos["Probabilidad de positivo"] >= umbral).map({True: "Quiebra", False: "Sana"}),
    })
    hojas[umbral] = hoja
    print(f"Umbral {umbral:.2f}: {(hoja['Clasificación'] == 'Quiebra').sum()} empresas marcadas como quiebra de {len(hoja)}")

Umbral 0.70: 94 empresas marcadas como quiebra de 1000
Umbral 0.35: 163 empresas marcadas como quiebra de 1000
Umbral 0.50: 134 empresas marcadas como quiebra de 1000


**Observación:** bajar el umbral agranda la lista de empresas a revisar. Cuántas revisar es una decisión de capacidad del equipo de riesgo y costo de equivocarse, no del modelo.

---
## 4. Exportar a Excel

In [6]:
with pd.ExcelWriter("predicciones_quiebra.xlsx") as writer:
    for umbral, hoja in hojas.items():
        hoja.to_excel(writer, sheet_name=f"Umbral {umbral:.2f}", index=False)

print("Archivo generado: predicciones_quiebra.xlsx")
pd.read_excel("predicciones_quiebra.xlsx", sheet_name=None)["Umbral 0.35"].sort_values(
    "Probabilidad de positivo", ascending=False
).head(10)

Archivo generado: predicciones_quiebra.xlsx


,Dato,Probabilidad de positivo,Clasificación
741,EMP-0742,0.9934,Quiebra
218,EMP-0219,0.9929,Quiebra
253,EMP-0254,0.9925,Quiebra
524,EMP-0525,0.9921,Quiebra
667,EMP-0668,0.9918,Quiebra
519,EMP-0520,0.9908,Quiebra
437,EMP-0438,0.9898,Quiebra
151,EMP-0152,0.9874,Quiebra
161,EMP-0162,0.9872,Quiebra
886,EMP-0887,0.9864,Quiebra
